# Station rainfall from the Rajasthan PDFs

**Alwar groundwater study · 14-block domain (LGD 574–587)**

Daily station rainfall from the Rajasthan rainfall PDFs (2010–2025, 16 files, 7,101 pages), kept only for stations that match groundwater block names, then de-duplicated.

| | |
|---|---|
| **Reads** | `gw_framework/data/Rainfall_Rajasthan/*.pdf`, `gw_framework/data/Rajasthan_gw_data/*.xlsx` (block names) |
| **Writes** | `Rainfall_Rajasthan/Matched_Block_Rainfall/matched_daily_rainfall.csv` (466,295 rows), `Rainfall_Rajasthan/Rainfall_Audit_Minimum_Conflict_Rule/` |
| **Cells** | 7 code cells, copied unchanged from `alwargwd_data_analysis_2_2.ipynb` (outputs cleared) |

**How to run:** run the set-up cell, then each section from top to bottom. Every cell reads its inputs from Drive, except where a label says it needs the cell above.

Cell labels: `[orig #N]` = position in the original notebook · ✅ ran OK · ⚠ last run ended with an error · ⏸ interrupted · • output not saved.

**Contents**
- 1. Inspect the PDFs
- 2. Extract daily rainfall
- 3. Duplicates and conflicts

In [ ]:
# ── SETUP — run this cell first ─────────────────────────────────────────────
import os, sys, json, subprocess
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

CFG = "/content/drive/MyDrive/gw_framework/repo_config.json"   # remembers your repository
if os.path.isfile(CFG):
    REPO = json.load(open(CFG))["repo"]
else:
    REPO = input("GitHub repository (username/repo-name): ").strip().strip("/")
    json.dump({"repo": REPO}, open(CFG, "w"))

REPO_DIR = "/content/" + REPO.split("/")[-1]
try:                                   # private repository: add a Colab secret named GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
elif subprocess.run(["git", "clone", "-q", URL, REPO_DIR]).returncode != 0:
    os.remove(CFG)
    raise RuntimeError(f"Could not clone '{REPO}'. Check the name (and GH_TOKEN if private), then run this cell again.")
sys.path.insert(0, REPO_DIR + "/src")

from alwar_gw import colab
PATHS = colab.setup(packages=["pdfplumber", "pymupdf"], modflow=False)
del TOKEN, URL


## 1. Inspect the PDFs

**[orig #62]** List the Rajasthan rainfall PDFs (2010–2025)  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

RAINFALL_DIR = Path(
    "/content/drive/MyDrive/gw_framework/data/Rainfall_Rajasthan"
)

print("Folder exists:", RAINFALL_DIR.exists())
print("Folder:", RAINFALL_DIR)

files = [p for p in RAINFALL_DIR.rglob("*") if p.is_file()]

print(f"\nTotal files: {len(files)}")

for f in files:
    print(f"\n{'='*70}")
    print("File:", f.relative_to(RAINFALL_DIR))
    print(f"Size: {f.stat().st_size / 1024:.1f} KB")

    try:
        ext = f.suffix.lower()

        if ext in [".xlsx", ".xls", ".xlsm"]:
            xls = pd.ExcelFile(f)
            print("Sheets:", xls.sheet_names)

            for sheet in xls.sheet_names:
                df = pd.read_excel(f, sheet_name=sheet)
                print(f"\nSheet: {sheet}")
                print("Shape:", df.shape)
                print("Columns:", df.columns.tolist())
                display(df.head(5))

        elif ext == ".csv":
            df = pd.read_csv(f, nrows=5)
            print("Columns:", df.columns.tolist())
            display(df)

        elif ext == ".parquet":
            df = pd.read_parquet(f)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            display(df.head(5))

        elif ext in [".nc", ".nc4"]:
            import xarray as xr
            ds = xr.open_dataset(f)
            print(ds)
            ds.close()

        else:
            print("Format not automatically previewed:", ext)

    except Exception as e:
        print("Inspection error:", repr(e))


**[orig #63]** Inspect PDF pages: text layer, rainfall keywords, sample tables  
<sub>✅ ran OK</sub>

In [ ]:

!pip -q install pymupdf pdfplumber

from pathlib import Path
import fitz
import pdfplumber
import pandas as pd
import re

RAINFALL_DIR = Path(
    "/content/drive/MyDrive/gw_framework/data/Rainfall_Rajasthan"
)

REPORT_DIR = RAINFALL_DIR / "PDF_Inspection"
REPORT_DIR.mkdir(exist_ok=True)

pdf_files = sorted(
    RAINFALL_DIR.glob("*.pdf"),
    key=lambda p: int(re.search(r"\d{4}", p.stem).group())
    if re.search(r"\d{4}", p.stem) else 9999
)

summary = []
page_details = []
sample_texts = []

for pdf_path in pdf_files:
    print("\n" + "=" * 90)
    print("FILE:", pdf_path.name)

    try:
        doc = fitz.open(pdf_path)
        page_count = len(doc)
        text_pages = 0
        total_chars = 0
        first_text = ""
        pages_with_rainfall_terms = []
        image_pages = 0

        # Inspect all pages for text availability and rainfall-related terms.
        for i, page in enumerate(doc):
            text = page.get_text("text").strip()
            total_chars += len(text)

            if len(text) > 30:
                text_pages += 1

            if i == 0:
                first_text = text[:5000]

            if page.get_images(full=True):
                image_pages += 1

            if re.search(
                r"rainfall|precipitation|district|station|gauge|"
                r"date|daily|mm\b|block",
                text,
                flags=re.I
            ):
                pages_with_rainfall_terms.append(i + 1)

            page_details.append({
                "file": pdf_path.name,
                "page": i + 1,
                "text_characters": len(text),
                "contains_images": bool(page.get_images(full=True)),
                "rainfall_related_terms": bool(re.search(
                    r"rainfall|precipitation|district|station|gauge|"
                    r"date|daily|mm\b|block",
                    text,
                    flags=re.I
                ))
            })

        doc.close()

        # Sample table extraction from up to 5 pages:
        # first page, middle page, last page and pages with relevant terms.
        candidates = sorted(set(
            [1, max(1, page_count // 2), page_count]
            + pages_with_rainfall_terms[:3]
        ))
        candidates = [p for p in candidates if 1 <= p <= page_count][:6]

        table_count = 0
        table_examples = []

        with pdfplumber.open(pdf_path) as pdf:
            for page_num in candidates:
                page = pdf.pages[page_num - 1]
                tables = page.extract_tables()

                if tables:
                    table_count += len(tables)
                    for table in tables[:2]:
                        table_examples.append({
                            "page": page_num,
                            "rows_detected": len(table),
                            "columns_detected": max(
                                [len(r) for r in table if r] or [0]
                            ),
                            "sample": str(table[:4])[:1500]
                        })

        summary.append({
            "file": pdf_path.name,
            "year": re.search(r"\d{4}", pdf_path.stem).group()
                    if re.search(r"\d{4}", pdf_path.stem) else None,
            "size_MB": round(pdf_path.stat().st_size / 1024**2, 2),
            "pages": page_count,
            "pages_with_searchable_text": text_pages,
            "total_extracted_characters": total_chars,
            "pages_with_images": image_pages,
            "pages_with_rainfall_terms": len(pages_with_rainfall_terms),
            "sampled_pages_with_tables": table_count
        })

        print("Pages:", page_count)
        print("Pages with searchable text:", text_pages)
        print("Pages containing images:", image_pages)
        print("Pages with relevant terms:", len(pages_with_rainfall_terms))
        print("\nFirst-page text sample:\n")
        print(first_text[:2500] if first_text else
              "[No extractable text: OCR may be needed]")

        print("\nSample extracted tables:")
        if table_examples:
            for item in table_examples:
                print(f"\nPage {item['page']}: "
                      f"{item['rows_detected']} rows, "
                      f"{item['columns_detected']} columns")
                print(item["sample"])
        else:
            print("No tables detected in sampled pages.")

        sample_texts.append({
            "file": pdf_path.name,
            "first_page_text": first_text
        })

    except Exception as e:
        print("ERROR:", repr(e))
        summary.append({
            "file": pdf_path.name,
            "error": str(e)
        })

summary_df = pd.DataFrame(summary)
pages_df = pd.DataFrame(page_details)
text_df = pd.DataFrame(sample_texts)

summary_df.to_csv(REPORT_DIR / "pdf_summary.csv", index=False)
pages_df.to_csv(REPORT_DIR / "pdf_page_details.csv", index=False)
text_df.to_csv(REPORT_DIR / "pdf_first_page_text.csv", index=False)

print("\n" + "=" * 90)
print("PDF SUMMARY")
display(summary_df)

print("\nReports saved to:", REPORT_DIR)
print("1. pdf_summary.csv")
print("2. pdf_page_details.csv")
print("3. pdf_first_page_text.csv")


**[orig #3]** Look at one sample page (2010.pdf) with pdfplumber  
<sub>✅ ran OK</sub>

In [ ]:

import pdfplumber
from pathlib import Path

pdf_path = Path(
    "/content/drive/MyDrive/gw_framework/data/Rainfall_Rajasthan/2010.pdf"
)

with pdfplumber.open(pdf_path) as pdf:
    # The first page appears to contain the AJMER station in your sample.
    page = pdf.pages[0]

    print("PAGE TEXT:")
    print((page.extract_text() or "")[:10000])

    print("\nTABLES DETECTED:")
    tables = page.extract_tables()

    print("Number of tables:", len(tables))

    for i, table in enumerate(tables):
        print(f"\nTABLE {i + 1}")
        for row in table[:12]:
            print(row)


## 2. Extract daily rainfall

Earlier versions of this extraction (original cells 1, 2, 4, 5) are in the archive notebook.

**[orig #6]** **MAIN** — extract daily station rainfall for stations matching groundwater block names → `Rainfall_Rajasthan/Matched_Block_Rainfall/`  
<sub>✅ ran OK</sub>

In [ ]:
# ================================================================
# COMPLETE SCRIPT: RAINFALL DATA FROM MATCHED GROUNDWATER BLOCKS ONLY
# Rajasthan | Google Colab
#
# Inputs:
#   gw_framework/data/Rajasthan_gw_data/*.xlsx
#   gw_framework/data/Rainfall_Rajasthan/*.pdf
#
# Output:
#   gw_framework/data/Rainfall_Rajasthan/Matched_Block_Rainfall/
#
# Rules:
# 1. Match PDF station names directly to groundwater Block names.
# 2. Matching is case-insensitive.
# 3. Ignore asterisks, punctuation, hyphens, underscores and extra spaces
#    during matching only.
# 4. Preserve the original station name in output.
# 5. Skip unmatched stations before extracting daily rainfall.
# 6. Do not use district names for matching.
# 7. Keep duplicate station-date records and flag them for review.
# 8. Validate extracted data against printed monthly/annual totals.
# ================================================================

import sys
import subprocess
import importlib.util
from pathlib import Path
import re
import calendar
import warnings

for module_name, package_name in {
    "pdfplumber": "pdfplumber",
    "pandas": "pandas",
    "openpyxl": "openpyxl",
}.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

import numpy as np
import pandas as pd
import pdfplumber

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------- 1. MOUNT DRIVE ------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Google Colab not detected; using the configured paths.")

# ------------------------- 2. PATHS ------------------------------

BASE_DIR = Path("/content/drive/MyDrive/gw_framework/data")

GROUNDWATER_DIR = BASE_DIR / "Rajasthan_gw_data"
RAINFALL_DIR = BASE_DIR / "Rainfall_Rajasthan"
OUTPUT_DIR = RAINFALL_DIR / "Matched_Block_Rainfall"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DAILY_OUTPUT = OUTPUT_DIR / "matched_daily_rainfall.csv"
VALIDATION_OUTPUT = OUTPUT_DIR / "matched_station_validation.csv"
ISSUES_OUTPUT = OUTPUT_DIR / "matched_station_issues.csv"
DUPLICATES_OUTPUT = OUTPUT_DIR / "matched_station_duplicates.csv"
SUMMARY_OUTPUT = OUTPUT_DIR / "processing_summary.txt"

MONTHLY_TOLERANCE_MM = 0.20
ANNUAL_TOLERANCE_MM = 0.50

if not GROUNDWATER_DIR.exists():
    raise FileNotFoundError(f"Groundwater folder not found: {GROUNDWATER_DIR}")

if not RAINFALL_DIR.exists():
    raise FileNotFoundError(f"Rainfall folder not found: {RAINFALL_DIR}")

pdf_files = sorted(RAINFALL_DIR.glob("*.pdf"))
excel_files = sorted(
    p for p in GROUNDWATER_DIR.glob("*.xlsx")
    if not p.name.startswith("~$")
)

if not pdf_files:
    raise FileNotFoundError(f"No PDFs found in {RAINFALL_DIR}")

if not excel_files:
    raise FileNotFoundError(f"No Excel files found in {GROUNDWATER_DIR}")

# ------------------------- 3. NAME NORMALIZATION -----------------

def normalize_name(value):
    """Normalize a name for matching without changing source data."""
    if value is None or pd.isna(value):
        return ""

    name = str(value).strip().lower()
    name = name.replace("*", " ")
    name = name.replace("_", " ").replace("-", " ")
    name = re.sub(r"[\u200b\u00a0]", " ", name)
    name = re.sub(r"[^\w\s]", "", name, flags=re.UNICODE)
    name = re.sub(r"\s+", " ", name).strip()

    return name


def compact_column_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


# ------------------------- 4. READ GROUNDWATER BLOCKS ------------

accepted_block_columns = {
    "block",
    "blockname",
    "tehsil",
    "tehsilname",
}

block_lookup = {}

for excel_path in excel_files:
    try:
        workbook = pd.ExcelFile(excel_path)
    except Exception as exc:
        print(f"WARNING: Cannot read {excel_path.name}: {exc}")
        continue

    for sheet_name in workbook.sheet_names:
        try:
            df = pd.read_excel(
                excel_path,
                sheet_name=sheet_name,
                dtype=str,
            )
        except Exception as exc:
            print(
                f"WARNING: Cannot read {excel_path.name}, "
                f"sheet {sheet_name}: {exc}"
            )
            continue

        if df.empty:
            continue

        block_column = next(
            (
                col for col in df.columns
                if compact_column_name(col) in accepted_block_columns
            ),
            None,
        )

        if block_column is None:
            continue

        for original_block in df[block_column].dropna().astype(str):
            original_block = original_block.strip()

            if not original_block:
                continue

            normalized = normalize_name(original_block)

            if normalized:
                block_lookup.setdefault(normalized, []).append({
                    "block_original": original_block,
                    "file": excel_path.name,
                    "sheet": sheet_name,
                })

if not block_lookup:
    raise ValueError(
        "No groundwater block names found. "
        "Check that your Excel files contain a Block or Tehsil column."
    )

print(f"Groundwater block names loaded: {len(block_lookup)}")
print(f"Rainfall PDFs found: {len(pdf_files)}")

# ------------------------- 5. PDF METADATA HELPERS ---------------

def extract_field(text, field_name, stop_fields):
    stops = "|".join(re.escape(s) for s in stop_fields)

    pattern = (
        rf"\b{re.escape(field_name)}\s*:\s*(.*?)"
        rf"(?=\s+(?:{stops})\s*:|$)"
    )

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if not match:
        return None

    value = re.sub(r"\s+", " ", match.group(1)).strip()
    return value or None


def infer_year(filename):
    matches = re.findall(r"\b(?:19|20)\d{2}\b", filename)
    return int(matches[-1]) if matches else None


MONTHS = [
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC",
]


def parse_rainfall_token(token):
    token = str(token).strip().upper()

    if token in {"TR", "TRACE", "T"}:
        return 0.0, "TRACE_VALUE_TREATED_AS_ZERO"

    if token in {"-", "--"}:
        return np.nan, "MISSING_VALUE"

    try:
        value = float(token)
    except ValueError:
        return np.nan, "INVALID_VALUE"

    if value < 0:
        return np.nan, "NEGATIVE_VALUE"

    return value, None


def extract_daily_table(text, year):
    """Extract daily rainfall only after a page has matched a block."""
    lines = text.splitlines()
    header_index = None

    for i, line in enumerate(lines):
        line_upper = re.sub(r"\s+", " ", line.strip().upper())

        if (
            re.search(r"\bDATE\b", line_upper)
            and all(month in line_upper for month in ["JAN", "FEB", "MAR", "DEC"])
        ):
            header_index = i
            break

    if header_index is None:
        return [], ["DAILY_TABLE_HEADER_NOT_FOUND"]

    records = []
    issues = []
    seen_days = set()
    table_started = False

    token_pattern = re.compile(
        r"(?<![A-Za-z0-9])"
        r"(?:\d+(?:\.\d*)?|\.\d+|TR|TRACE|T|--|-)"
        r"(?![A-Za-z0-9])",
        flags=re.IGNORECASE,
    )

    for line_index in range(header_index + 1, len(lines)):
        line = lines[line_index].strip()

        if not line:
            continue

        if re.search(r"\bMONTHLY\s+RAINFALL\b", line, re.IGNORECASE):
            break

        row_match = re.match(r"^\s*(\d{1,2})\s+(.*)$", line)

        if not row_match:
            continue

        day = int(row_match.group(1))
        remainder = row_match.group(2)

        if not 1 <= day <= 31:
            continue

        tokens = token_pattern.findall(remainder)

        if not tokens:
            continue

        table_started = True

        if day in seen_days:
            issues.append(f"DUPLICATE_DAILY_ROW_FOR_DAY_{day}")
            continue

        seen_days.add(day)

        valid_months = [
            month for month in range(1, 13)
            if day <= calendar.monthrange(year, month)[1]
        ]

        if len(tokens) != len(valid_months):
            issues.append(
                f"DAY_{day}_VALUE_COUNT_MISMATCH_"
                f"EXPECTED_{len(valid_months)}_FOUND_{len(tokens)}"
            )
            continue

        parsed_values = []
        row_invalid = False

        for token in tokens:
            value, issue = parse_rainfall_token(token)

            if issue == "TRACE_VALUE_TREATED_AS_ZERO":
                issues.append(f"TRACE_VALUE_DAY_{day}_{token}")
            elif issue is not None:
                issues.append(f"{issue}_DAY_{day}_{token}")
                row_invalid = True

            parsed_values.append(value)

        if row_invalid:
            continue

        for month, rainfall_mm in zip(valid_months, parsed_values):
            records.append({
                "year": year,
                "month": month,
                "month_name": calendar.month_abbr[month].upper(),
                "day": day,
                "date": f"{year:04d}-{month:02d}-{day:02d}",
                "rainfall_mm": rainfall_mm,
            })

    if not table_started:
        issues.append("NO_DAILY_ROWS_FOUND")

    return records, issues


def extract_monthly_summary(text):
    match = re.search(
        r"\bMONTHLY\s+RAINFALL\b(.*?)\bRAINY\s*DAYS\b",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if not match:
        match = re.search(
            r"\bMONTHLY\s+RAINFALL\b(.*?)\bRAINYDAYS\b",
            text,
            flags=re.IGNORECASE | re.DOTALL,
        )

    if not match:
        return None, "MONTHLY_SUMMARY_NOT_FOUND"

    token_pattern = re.compile(
        r"(?<![A-Za-z0-9])"
        r"(?:\d+(?:\.\d*)?|\.\d+|TR|TRACE|T|--|-)"
        r"(?![A-Za-z0-9])",
        flags=re.IGNORECASE,
    )

    tokens = token_pattern.findall(match.group(1))
    values = []

    for token in tokens:
        value, issue = parse_rainfall_token(token)

        if issue in {"MISSING_VALUE", "INVALID_VALUE", "NEGATIVE_VALUE"}:
            return None, f"INVALID_MONTHLY_SUMMARY_TOKEN_{token}"

        values.append(value)

    if len(values) != 12:
        return None, f"MONTHLY_SUMMARY_EXPECTED_12_VALUES_FOUND_{len(values)}"

    return values, None


def extract_annual_summary(text):
    patterns = [
        r"\bANNUAL\s+RAINFALL\b\s*[:=\-]?\s*(\d+(?:\.\d*)?|\.\d+)",
        r"\bYEARLY\s+RAINFALL\b\s*[:=\-]?\s*(\d+(?:\.\d*)?|\.\d+)",
        r"\bTOTAL\s+RAINFALL\b\s*[:=\-]?\s*(\d+(?:\.\d*)?|\.\d+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return float(match.group(1)), None

    return None, "ANNUAL_SUMMARY_NOT_FOUND"


# ------------------------- 6. EXTRACT MATCHED STATIONS ONLY -------

daily_records = []
validation_records = []
issue_records = []

matched_pages = 0
skipped_pages = 0
pages_processed = 0

for pdf_path in pdf_files:
    year_from_filename = infer_year(pdf_path.name)
    print(f"Checking: {pdf_path.name}")

    try:
        pdf = pdfplumber.open(pdf_path)
    except Exception as exc:
        print(f"  WARNING: PDF could not be opened: {exc}")
        continue

    with pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            pages_processed += 1

            try:
                text = page.extract_text(
                    x_tolerance=2,
                    y_tolerance=3,
                ) or ""
            except Exception as exc:
                continue

            if not text.strip():
                continue

            station = extract_field(
                text,
                "STATION",
                [
                    "DISTRICT", "STATE", "CATCHMENT",
                    "LATITUDE", "LONGITUDE", "BLOCK",
                ],
            )

            if not station:
                continue

            station_original = station.strip()
            station_normalized = normalize_name(station_original)

            # IMPORTANT:
            # Unmatched station pages are skipped BEFORE daily table parsing.
            if station_normalized not in block_lookup:
                skipped_pages += 1
                continue

            matched_pages += 1

            year = year_from_filename

            if year is None:
                year_match = re.search(
                    r"\bYEAR\s*:?\s*((?:19|20)\d{2})\b",
                    text,
                    flags=re.IGNORECASE,
                )
                if year_match:
                    year = int(year_match.group(1))

            matched_block_names = sorted({
                item["block_original"]
                for item in block_lookup[station_normalized]
            })

            matched_block_files = sorted({
                item["file"]
                for item in block_lookup[station_normalized]
            })

            page_info = {
                "pdf_file": pdf_path.name,
                "page_number": page_number,
                "year": year,
                "station_original": station_original,
                "station_normalized_for_matching": station_normalized,
                "groundwater_block_match": "; ".join(matched_block_names),
                "groundwater_files": "; ".join(matched_block_files),
            }

            if year is None:
                validation_records.append({
                    **page_info,
                    "daily_records_extracted": 0,
                    "calculated_annual_mm": np.nan,
                    "printed_annual_mm": np.nan,
                    "annual_difference_mm": np.nan,
                    "monthly_validation": "NOT_CHECKED",
                    "annual_validation": "NOT_CHECKED",
                    "overall_status": "YEAR_NOT_DETERMINED",
                })
                issue_records.append({
                    **page_info,
                    "issue": "YEAR_NOT_DETERMINED",
                })
                continue

            daily_page, daily_issues = extract_daily_table(text, year)

            monthly_summary, monthly_issue = extract_monthly_summary(text)
            printed_annual, annual_issue = extract_annual_summary(text)

            for record in daily_page:
                daily_records.append({
                    **record,
                    "station_original": station_original,
                    "station_normalized_for_matching": station_normalized,
                    "groundwater_block_match": "; ".join(matched_block_names),
                    "pdf_file": pdf_path.name,
                    "page_number": page_number,
                })

            for issue in daily_issues:
                issue_records.append({
                    **page_info,
                    "issue": issue,
                })

            if monthly_issue:
                issue_records.append({
                    **page_info,
                    "issue": monthly_issue,
                })

            if annual_issue:
                issue_records.append({
                    **page_info,
                    "issue": annual_issue,
                })

            calculated_annual = (
                float(sum(r["rainfall_mm"] for r in daily_page))
                if daily_page else np.nan
            )

            # Monthly validation compares extracted daily sums with the
            # corresponding monthly totals printed on the PDF page.
            monthly_status = "NOT_CHECKED"

            if daily_page and monthly_summary is not None:
                daily_page_df = pd.DataFrame(daily_page)
                differences = []

                for month_number in range(1, 13):
                    month_values = daily_page_df.loc[
                        daily_page_df["month"] == month_number,
                        "rainfall_mm",
                    ]

                    if month_values.empty:
                        differences.append(np.nan)
                    else:
                        differences.append(
                            float(month_values.sum())
                            - float(monthly_summary[month_number - 1])
                        )

                if all(pd.notna(d) for d in differences):
                    monthly_status = (
                        "PASS"
                        if all(
                            abs(d) <= MONTHLY_TOLERANCE_MM
                            for d in differences
                        )
                        else "FAIL"
                    )

                    for month_number, difference in enumerate(differences, 1):
                        if abs(difference) > MONTHLY_TOLERANCE_MM:
                            issue_records.append({
                                **page_info,
                                "issue": (
                                    f"MONTHLY_TOTAL_MISMATCH_"
                                    f"{MONTHS[month_number - 1]}_"
                                    f"DIFFERENCE_{difference:.3f}_MM"
                                ),
                            })
                else:
                    monthly_status = "INCOMPLETE_DAILY_MONTH_DATA"

            elif monthly_summary is None:
                monthly_status = "SUMMARY_NOT_AVAILABLE"
            else:
                monthly_status = "DAILY_DATA_NOT_AVAILABLE"

            # Annual validation.
            annual_difference = np.nan

            if daily_page and printed_annual is not None:
                annual_difference = calculated_annual - printed_annual
                annual_status = (
                    "PASS"
                    if abs(annual_difference) <= ANNUAL_TOLERANCE_MM
                    else "FAIL"
                )

                if annual_status == "FAIL":
                    issue_records.append({
                        **page_info,
                        "issue": (
                            f"ANNUAL_TOTAL_MISMATCH_CALCULATED_"
                            f"{calculated_annual:.3f}_PRINTED_"
                            f"{printed_annual:.3f}_"
                            f"DIFFERENCE_{annual_difference:.3f}_MM"
                        ),
                    })
            elif printed_annual is None:
                annual_status = "SUMMARY_NOT_AVAILABLE"
            else:
                annual_status = "DAILY_DATA_NOT_AVAILABLE"

            if not daily_page:
                overall_status = "FAIL_NO_DAILY_DATA"
            elif monthly_status == "FAIL" or annual_status == "FAIL":
                overall_status = "FAIL_VALIDATION_MISMATCH"
            elif (
                monthly_status == "PASS"
                and annual_status == "PASS"
                and not daily_issues
            ):
                overall_status = "PASS"
            else:
                overall_status = "REVIEW"

            validation_records.append({
                **page_info,
                "daily_records_extracted": len(daily_page),
                "calculated_annual_mm": calculated_annual,
                "printed_annual_mm": (
                    printed_annual if printed_annual is not None else np.nan
                ),
                "annual_difference_mm": annual_difference,
                "monthly_validation": monthly_status,
                "annual_validation": annual_status,
                "overall_status": overall_status,
            })

# ------------------------- 7. SAVE MATCHED DATA ONLY --------------

daily_df = pd.DataFrame(daily_records)
validation_df = pd.DataFrame(validation_records)
issues_df = pd.DataFrame(issue_records)

if not daily_df.empty:
    daily_df = daily_df.sort_values(
        [
            "station_normalized_for_matching",
            "date",
            "pdf_file",
            "page_number",
        ],
        kind="stable",
    )

daily_df.to_csv(
    DAILY_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

validation_df.to_csv(
    VALIDATION_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

issues_df.to_csv(
    ISSUES_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

# ------------------------- 8. DUPLICATE REVIEW -------------------

if not daily_df.empty:
    duplicate_mask = daily_df.duplicated(
        subset=["station_normalized_for_matching", "date"],
        keep=False,
    )

    duplicates_df = daily_df.loc[duplicate_mask].copy()
else:
    duplicates_df = pd.DataFrame(columns=daily_df.columns)

duplicates_df.to_csv(
    DUPLICATES_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

# ------------------------- 9. PROCESSING SUMMARY -----------------

if not daily_df.empty:
    unique_stations = daily_df[
        "station_normalized_for_matching"
    ].nunique()

    unique_dates = daily_df["date"].nunique()
    earliest_date = daily_df["date"].min()
    latest_date = daily_df["date"].max()
else:
    unique_stations = 0
    unique_dates = 0
    earliest_date = None
    latest_date = None

if not validation_df.empty:
    status_counts = validation_df["overall_status"].value_counts().to_dict()
else:
    status_counts = {}

summary = [
    "MATCHED-STATION RAINFALL EXTRACTION SUMMARY",
    "=" * 50,
    f"Groundwater input folder: {GROUNDWATER_DIR}",
    f"Rainfall input folder: {RAINFALL_DIR}",
    f"Output folder: {OUTPUT_DIR}",
    "",
    f"PDF files found: {len(pdf_files)}",
    f"Groundwater Excel files found: {len(excel_files)}",
    f"Groundwater block names loaded: {len(block_lookup)}",
    f"PDF pages processed: {pages_processed}",
    f"Matched station pages processed: {matched_pages}",
    f"Unmatched station pages skipped: {skipped_pages}",
    f"Daily rainfall records saved: {len(daily_df)}",
    f"Unique matched stations: {unique_stations}",
    f"Unique dates: {unique_dates}",
    f"First date: {earliest_date}",
    f"Last date: {latest_date}",
    f"Validation status counts: {status_counts}",
    f"Rows flagged for duplicate station-date review: {len(duplicates_df)}",
    "",
    "OUTPUT FILES",
    f"Matched daily rainfall: {DAILY_OUTPUT}",
    f"Matched station validation: {VALIDATION_OUTPUT}",
    f"Matched station issues: {ISSUES_OUTPUT}",
    f"Matched station duplicates: {DUPLICATES_OUTPUT}",
    "",
    "Only rainfall from station pages matching groundwater block names",
    "is extracted. Unmatched station pages are skipped and are not saved.",
    "Review validation failures and duplicates before modelling.",
]

SUMMARY_OUTPUT.write_text("\n".join(summary), encoding="utf-8")

print("\n" + "=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)

for line in summary:
    print(line)

print("\nPreview of matched rainfall records:")
if daily_df.empty:
    print(
        "No matching daily records were extracted. "
        "Check the station naming, PDF text layout, and block column names."
    )
else:
    print(daily_df.head(10).to_string(index=False))

print("\nValidation statuses:")
if not validation_df.empty:
    print(validation_df["overall_status"].value_counts().to_string())
else:
    print("No matched station pages were validated.")

print("\nOnly the four output files listed above are saved in the output folder.")

## 3. Duplicates and conflicts

Result: 427,941 station-days; 5,455 had conflicting values, resolved by the minimum-conflict rule.

**[orig #7]** Duplicate audit and quality control of the extracted rainfall  
<sub>✅ ran OK</sub>

In [ ]:
# =====================================================================
# COMPLETE RAINFALL DUPLICATE AUDIT AND QUALITY CONTROL
# Rajasthan groundwater PhD research
#
# INPUT FILES (original files are READ ONLY):
#   Matched_Block_Rainfall/matched_daily_rainfall.csv
#   Matched_Block_Rainfall/matched_station_validation.csv
#   Matched_Block_Rainfall/matched_station_issues.csv
#
# OUTPUT FOLDER:
#   Rainfall_Rajasthan/Rainfall_Audit/
#
# OUTPUTS:
#   1. audited_station_daily_rainfall.csv
#   2. recommended_model_ready_rainfall.csv
#   3. exact_duplicate_groups.csv
#   4. conflicting_station_date_records.csv
#   5. source_page_audit.csv
#   6. validation_failure_pages.csv
#   7. audit_summary.txt
#
# POLICY:
# - Do not modify original files.
# - Consolidate records only when station, date and rainfall agree.
# - Do not average conflicting rainfall values.
# - Conflicting station-date groups are excluded from the model-ready file.
# - Model-ready data require a unique rainfall value and PASS validation
#   for every source page contributing to that station-date group.
# - Keep an audit trail for all decisions.
# =====================================================================

import sys
import subprocess
import importlib.util
from pathlib import Path
import re
import warnings

# ------------------------- 1. INSTALL PACKAGES ------------------------

for module_name, package_name in {
    "pandas": "pandas",
    "numpy": "numpy",
}.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------- 2. MOUNT GOOGLE DRIVE ----------------------

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Google Colab not detected; using configured file paths.")

# ------------------------- 3. FILE PATHS ------------------------------

BASE_DIR = Path("/content/drive/MyDrive/gw_framework/data")

RAINFALL_DIR = BASE_DIR / "Rainfall_Rajasthan"
INPUT_DIR = RAINFALL_DIR / "Matched_Block_Rainfall"
OUTPUT_DIR = RAINFALL_DIR / "Rainfall_Audit"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DAILY_INPUT = INPUT_DIR / "matched_daily_rainfall.csv"
VALIDATION_INPUT = INPUT_DIR / "matched_station_validation.csv"
ISSUES_INPUT = INPUT_DIR / "matched_station_issues.csv"

AUDITED_OUTPUT = OUTPUT_DIR / "audited_station_daily_rainfall.csv"
MODEL_READY_OUTPUT = OUTPUT_DIR / "recommended_model_ready_rainfall.csv"
EXACT_DUPLICATES_OUTPUT = OUTPUT_DIR / "exact_duplicate_groups.csv"
CONFLICTS_OUTPUT = OUTPUT_DIR / "conflicting_station_date_records.csv"
PAGE_AUDIT_OUTPUT = OUTPUT_DIR / "source_page_audit.csv"
FAILED_PAGES_OUTPUT = OUTPUT_DIR / "validation_failure_pages.csv"
SUMMARY_OUTPUT = OUTPUT_DIR / "audit_summary.txt"

# Values that differ by no more than this tolerance are treated as
# equivalent for duplicate-value comparison. Use 0.001 mm to avoid
# hiding meaningful differences in rainfall measurements.
VALUE_TOLERANCE_MM = 0.001

# ------------------------- 4. CHECK INPUTS ----------------------------

for input_path in [DAILY_INPUT, VALIDATION_INPUT, ISSUES_INPUT]:
    if not input_path.exists():
        raise FileNotFoundError(
            f"Required input file not found:\n{input_path}\n\n"
            "Check that the previous rainfall extraction script "
            "completed successfully."
        )

print("=" * 72)
print("RAINFALL DUPLICATE AND VALIDATION AUDIT")
print("=" * 72)
print(f"Daily input:      {DAILY_INPUT}")
print(f"Validation input: {VALIDATION_INPUT}")
print(f"Issues input:     {ISSUES_INPUT}")
print(f"Output folder:    {OUTPUT_DIR}")
print()

# ------------------------- 5. LOAD DATA -------------------------------

daily = pd.read_csv(DAILY_INPUT, low_memory=False)
validation = pd.read_csv(VALIDATION_INPUT, low_memory=False)
issues = pd.read_csv(ISSUES_INPUT, low_memory=False)

if daily.empty:
    raise ValueError(
        "The daily rainfall input file is empty. "
        "No audit can be performed."
    )

required_daily_columns = [
    "date",
    "rainfall_mm",
    "station_original",
]

missing_columns = [
    col for col in required_daily_columns
    if col not in daily.columns
]

if missing_columns:
    raise ValueError(
        f"Daily rainfall file is missing required columns: {missing_columns}"
    )

print(f"Daily rainfall rows loaded: {len(daily):,}")
print(f"Validation rows loaded:     {len(validation):,}")
print(f"Issue rows loaded:          {len(issues):,}")
print()

# ------------------------- 6. STANDARDIZE AUDIT KEYS ------------------

def normalize_name(value):
    """Use the same matching principles as the extraction script."""
    if pd.isna(value):
        return ""

    name = str(value).strip().lower()
    name = name.replace("*", " ")
    name = name.replace("_", " ").replace("-", " ")
    name = re.sub(r"[\u200b\u00a0]", " ", name)
    name = re.sub(r"[^\w\s]", "", name, flags=re.UNICODE)
    name = re.sub(r"\s+", " ", name).strip()

    return name


# Keep original station values unchanged.
daily["station_original"] = daily["station_original"].astype(str)

if "station_normalized_for_matching" not in daily.columns:
    daily["station_normalized_for_matching"] = (
        daily["station_original"].apply(normalize_name)
    )
else:
    # Recompute the key consistently; do not modify the original name.
    daily["station_normalized_for_matching"] = (
        daily["station_original"].apply(normalize_name)
    )

daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily["rainfall_mm"] = pd.to_numeric(
    daily["rainfall_mm"],
    errors="coerce",
)

if "year" in daily.columns:
    daily["year"] = pd.to_numeric(daily["year"], errors="coerce")

if "month" in daily.columns:
    daily["month"] = pd.to_numeric(daily["month"], errors="coerce")

if "day" in daily.columns:
    daily["day"] = pd.to_numeric(daily["day"], errors="coerce")

# Retain source-file and source-page information for traceability.
for col in ["pdf_file", "page_number"]:
    if col not in daily.columns:
        daily[col] = np.nan

daily["source_page_key"] = (
    daily["pdf_file"].fillna("").astype(str)
    + "::PAGE::"
    + daily["page_number"].fillna("").astype(str)
)

# Invalid keys or missing numeric rainfall cannot enter the audited
# unique-value dataset. They are retained in a dedicated issue report.
invalid_key_mask = (
    daily["date"].isna()
    | daily["station_normalized_for_matching"].eq("")
    | daily["rainfall_mm"].isna()
    | ~np.isfinite(daily["rainfall_mm"])
)

invalid_rows = daily.loc[invalid_key_mask].copy()
daily_valid = daily.loc[~invalid_key_mask].copy()

if daily_valid.empty:
    raise ValueError(
        "No valid station-date-rainfall records remain after checking "
        "dates, station names and numeric rainfall values."
    )

# ------------------------- 7. PREPARE PAGE VALIDATION -----------------

validation_required = {
    "pdf_file",
    "page_number",
    "overall_status",
}

if validation.empty:
    validation_lookup = {}
else:
    missing_validation_columns = validation_required - set(validation.columns)

    if missing_validation_columns:
        raise ValueError(
            "Validation file is missing required columns: "
            f"{sorted(missing_validation_columns)}"
        )

    validation = validation.copy()

    validation["pdf_file"] = validation["pdf_file"].fillna("").astype(str)
    validation["page_number_key"] = (
        pd.to_numeric(validation["page_number"], errors="coerce")
        .astype("Int64")
        .astype(str)
    )

    validation["source_page_key"] = (
        validation["pdf_file"]
        + "::PAGE::"
        + validation["page_number_key"]
    )

    validation["overall_status"] = (
        validation["overall_status"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # If a source page appears more than once in validation, treat its
    # combined status conservatively.
    status_priority = {
        "PASS": 0,
        "REVIEW": 1,
        "NOT_CHECKED": 2,
        "YEAR_NOT_DETERMINED": 3,
        "FAIL_NO_DAILY_DATA": 4,
        "FAIL_VALIDATION_MISMATCH": 5,
    }

    def combine_page_status(status_values):
        values = [str(v).upper() for v in status_values if pd.notna(v)]

        if not values:
            return "UNKNOWN"

        # Any failure means that the source page is not PASS.
        failures = [v for v in values if v.startswith("FAIL")]
        if failures:
            return sorted(
                failures,
                key=lambda x: status_priority.get(x, 99),
            )[-1]

        if all(v == "PASS" for v in values):
            return "PASS"

        non_pass = [v for v in values if v != "PASS"]

        return sorted(
            non_pass,
            key=lambda x: status_priority.get(x, 99),
        )[-1] if non_pass else "UNKNOWN"

    validation_lookup = (
        validation.groupby("source_page_key")["overall_status"]
        .apply(combine_page_status)
        .to_dict()
    )

daily_valid["source_page_status"] = (
    daily_valid["source_page_key"]
    .map(validation_lookup)
    .fillna("UNKNOWN")
)

daily_valid["source_page_pass"] = (
    daily_valid["source_page_status"].eq("PASS")
)

# ------------------------- 8. PAGE-LEVEL AUDIT ------------------------

page_audit = (
    daily_valid.groupby(
        [
            "station_normalized_for_matching",
            "date",
            "source_page_key",
            "pdf_file",
            "page_number",
            "source_page_status",
        ],
        dropna=False,
    )
    .agg(
        daily_rows_on_page=("rainfall_mm", "size"),
        unique_rainfall_values_on_page=(
            "rainfall_mm",
            lambda x: x.round(3).nunique(),
        ),
        minimum_rainfall_mm=("rainfall_mm", "min"),
        maximum_rainfall_mm=("rainfall_mm", "max"),
    )
    .reset_index()
)

page_audit.to_csv(
    PAGE_AUDIT_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

# ------------------------- 9. IDENTIFY DUPLICATE GROUPS ---------------

group_columns = [
    "station_normalized_for_matching",
    "date",
]

group_results = []
exact_duplicate_groups = []
conflict_records = []
audited_records = []
model_ready_records = []

# A group is consolidated only if its rainfall values agree within the
# stated tolerance. A conflict is never averaged or arbitrarily resolved.

for (station_key, obs_date), group in daily_valid.groupby(
    group_columns,
    sort=True,
    dropna=False,
):
    group = group.copy()

    rainfall_values = group["rainfall_mm"].to_numpy(dtype=float)
    sorted_values = np.sort(rainfall_values)

    # Cluster equivalent values using the tolerance.
    # A group is considered conflict-free only when the full range fits
    # within the tolerance. This avoids chain-merging several different
    # rainfall values through intermediate values.
    value_range = float(np.max(sorted_values) - np.min(sorted_values))
    values_agree = value_range <= VALUE_TOLERANCE_MM

    unique_source_pages = sorted(
        group["source_page_key"].dropna().astype(str).unique().tolist()
    )

    source_page_statuses = sorted(
        group["source_page_status"].fillna("UNKNOWN").astype(str).unique().tolist()
    )

    all_pages_pass = (
        len(source_page_statuses) > 0
        and all(status == "PASS" for status in source_page_statuses)
    )

    source_pdfs = sorted(
        group["pdf_file"].dropna().astype(str).unique().tolist()
    )

    original_station_names = sorted(
        group["station_original"].dropna().astype(str).unique().tolist()
    )

    rainfall_value_strings = sorted({
        f"{v:.6f}" for v in rainfall_values
    })

    record = {
        "station_normalized_for_matching": station_key,
        "date": pd.Timestamp(obs_date).strftime("%Y-%m-%d"),
        "station_original_values": "; ".join(original_station_names),
        "rainfall_mm": (
            float(np.median(rainfall_values))
            if values_agree else np.nan
        ),
        "source_row_count": len(group),
        "source_page_count": len(unique_source_pages),
        "source_pdfs": "; ".join(source_pdfs),
        "source_page_keys": "; ".join(unique_source_pages),
        "source_page_statuses": "; ".join(source_page_statuses),
        "all_source_pages_pass": all_pages_pass,
        "rainfall_value_range_mm": value_range,
        "rainfall_values_mm": "; ".join(rainfall_value_strings),
        "duplicate_group": len(group) > 1,
        "conflicting_rainfall_values": not values_agree,
    }

    if values_agree:
        record["audit_decision"] = (
            "UNIQUE_RECORD"
            if len(group) == 1
            else "CONSOLIDATED_AGREEING_DUPLICATES"
        )
    else:
        record["audit_decision"] = "EXCLUDED_CONFLICTING_VALUES"

    audited_records.append(record)

    group_results.append({
        "station_normalized_for_matching": station_key,
        "date": pd.Timestamp(obs_date).strftime("%Y-%m-%d"),
        "source_row_count": len(group),
        "source_page_count": len(unique_source_pages),
        "rainfall_values_mm": "; ".join(rainfall_value_strings),
        "rainfall_value_range_mm": value_range,
        "all_source_pages_pass": all_pages_pass,
        "source_page_statuses": "; ".join(source_page_statuses),
        "source_pdfs": "; ".join(source_pdfs),
        "audit_decision": record["audit_decision"],
    })

    if len(group) > 1 and values_agree:
        exact_duplicate_groups.append({
            "station_normalized_for_matching": station_key,
            "date": pd.Timestamp(obs_date).strftime("%Y-%m-%d"),
            "record_count": len(group),
            "source_page_count": len(unique_source_pages),
            "rainfall_value_mm": float(np.median(rainfall_values)),
            "rainfall_value_range_mm": value_range,
            "source_pdfs": "; ".join(source_pdfs),
            "source_page_keys": "; ".join(unique_source_pages),
            "source_page_statuses": "; ".join(source_page_statuses),
            "all_source_pages_pass": all_pages_pass,
            "decision": (
                "CONSOLIDATED"
                if all_pages_pass
                else "CONSOLIDATED_BUT_VALIDATION_REVIEW_REQUIRED"
            ),
        })

    if not values_agree:
        # Preserve every original row in conflicting groups.
        conflict_subset = group.copy()
        conflict_subset["conflict_reason"] = (
            "Different rainfall values for the same normalized station/date"
        )
        conflict_subset["group_rainfall_range_mm"] = value_range
        conflict_records.append(conflict_subset)

    # A model-ready record requires agreement and PASS validation
    # for every contributing source page.
    if values_agree and all_pages_pass:
        model_ready_records.append({
            "station_normalized_for_matching": station_key,
            "date": pd.Timestamp(obs_date).strftime("%Y-%m-%d"),
            "rainfall_mm": float(np.median(rainfall_values)),
            "source_row_count": len(group),
            "source_page_count": len(unique_source_pages),
            "source_pdfs": "; ".join(source_pdfs),
            "source_page_keys": "; ".join(unique_source_pages),
            "quality_status": (
                "PASS_UNIQUE"
                if len(group) == 1
                else "PASS_DUPLICATES_AGREE_CONSOLIDATED"
            ),
        })

# ------------------------- 10. BUILD OUTPUT TABLES --------------------

audited_df = pd.DataFrame(audited_records)
exact_duplicates_df = pd.DataFrame(exact_duplicate_groups)

if conflict_records:
    conflicts_df = pd.concat(
        conflict_records,
        ignore_index=True,
    )
else:
    conflicts_df = pd.DataFrame(columns=list(daily_valid.columns) + [
        "conflict_reason",
        "group_rainfall_range_mm",
    ])

model_ready_df = pd.DataFrame(model_ready_records)
group_audit_df = pd.DataFrame(group_results)

# Save a separate invalid-row report inside the audit folder.
INVALID_ROWS_OUTPUT = OUTPUT_DIR / "invalid_input_rows.csv"
invalid_rows.to_csv(
    INVALID_ROWS_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

audited_df.to_csv(
    AUDITED_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

model_ready_df.to_csv(
    MODEL_READY_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

exact_duplicates_df.to_csv(
    EXACT_DUPLICATES_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

conflicts_df.to_csv(
    CONFLICTS_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

# ------------------------- 11. VALIDATION FAILURE REPORT --------------

if not validation.empty:
    failed_pages_df = validation.loc[
        validation["overall_status"].ne("PASS")
    ].copy()
else:
    failed_pages_df = pd.DataFrame()

failed_pages_df.to_csv(
    FAILED_PAGES_OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

# ------------------------- 12. SUMMARY STATISTICS --------------------

input_rows = len(daily)
valid_input_rows = len(daily_valid)
invalid_input_rows = len(invalid_rows)

unique_station_date_groups = len(audited_df)

exact_duplicate_group_count = len(exact_duplicates_df)

conflict_group_count = (
    conflicts_df[
        ["station_normalized_for_matching", "date"]
    ].drop_duplicates().shape[0]
    if not conflicts_df.empty
    else 0
)

model_ready_rows = len(model_ready_df)

if not model_ready_df.empty:
    model_ready_station_count = (
        model_ready_df["station_normalized_for_matching"].nunique()
    )
    model_ready_date_min = model_ready_df["date"].min()
    model_ready_date_max = model_ready_df["date"].max()
else:
    model_ready_station_count = 0
    model_ready_date_min = None
    model_ready_date_max = None

page_status_counts = (
    validation["overall_status"].value_counts(dropna=False).to_dict()
    if not validation.empty
    else {}
)

if not audited_df.empty:
    decision_counts = audited_df["audit_decision"].value_counts().to_dict()
else:
    decision_counts = {}

summary_lines = [
    "RAINFALL DUPLICATE AUDIT AND QUALITY CONTROL SUMMARY",
    "=" * 64,
    f"Original daily file: {DAILY_INPUT}",
    f"Output directory: {OUTPUT_DIR}",
    "",
    "INPUT COUNTS",
    f"Original daily rows: {input_rows:,}",
    f"Rows with valid station/date/rainfall: {valid_input_rows:,}",
    f"Invalid rows separated for review: {invalid_input_rows:,}",
    f"Validation pages: {len(validation):,}",
    "",
    "DUPLICATE AUDIT",
    f"Unique station-date groups audited: {unique_station_date_groups:,}",
    f"Groups with agreeing duplicate values: {exact_duplicate_group_count:,}",
    f"Groups with conflicting rainfall values: {conflict_group_count:,}",
    f"Audit decision counts: {decision_counts}",
    "",
    "CONSERVATIVE MODEL-READY DATASET",
    f"Station-date records: {model_ready_rows:,}",
    f"Unique stations: {model_ready_station_count:,}",
    f"First date: {model_ready_date_min}",
    f"Last date: {model_ready_date_max}",
    "",
    "SOURCE VALIDATION STATUS COUNTS",
    str(page_status_counts),
    "",
    "OUTPUT FILES",
    f"Audited station-date data: {AUDITED_OUTPUT}",
    f"Recommended model-ready data: {MODEL_READY_OUTPUT}",
    f"Agreeing duplicate groups: {EXACT_DUPLICATES_OUTPUT}",
    f"Conflicting records: {CONFLICTS_OUTPUT}",
    f"Source page audit: {PAGE_AUDIT_OUTPUT}",
    f"Validation failure/review pages: {FAILED_PAGES_OUTPUT}",
    f"Invalid input rows: {INVALID_ROWS_OUTPUT}",
    "",
    "INTERPRETATION",
    "The audited dataset contains one row per normalized station/date.",
    "Groups with conflicting rainfall values have no selected rainfall value.",
    "The model-ready dataset includes only groups with agreeing rainfall",
    "values and PASS validation for all contributing source pages.",
    "This is a conservative screening rule; excluded records should be",
    "reviewed before deciding whether they can be recovered.",
    "No original input file was modified.",
]

SUMMARY_OUTPUT.write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

# ------------------------- 13. DISPLAY RESULTS ------------------------

print("\n" + "=" * 72)
print("AUDIT COMPLETE")
print("=" * 72)

for line in summary_lines:
    print(line)

print("\nPreview: recommended model-ready rainfall")
if model_ready_df.empty:
    print(
        "No records meet the conservative model-ready criteria. "
        "Inspect the validation and conflict reports."
    )
else:
    print(model_ready_df.head(10).to_string(index=False))

print("\nPreview: conflicting station-date groups")
if conflicts_df.empty:
    print("No conflicting rainfall values were detected.")
else:
    preview_columns = [
        c for c in [
            "station_original",
            "station_normalized_for_matching",
            "date",
            "rainfall_mm",
            "pdf_file",
            "page_number",
            "source_page_status",
            "conflict_reason",
        ]
        if c in conflicts_df.columns
    ]
    print(conflicts_df[preview_columns].head(15).to_string(index=False))

print("\nAll audit outputs are saved separately.")
print("Your original matched_daily_rainfall.csv remains unchanged.")

**[orig #8]** **MAIN** — resolve conflicting station-date values (minimum-conflict rule) → `Rainfall_Audit_Minimum_Conflict_Rule/`  
<sub>✅ ran OK</sub>

In [ ]:
# =============================================================================
# RAJASTHAN GROUNDWATER PhD
# DAILY RAINFALL CONFLICT RESOLUTION AND AUDIT
#
# INPUT:
#   matched_daily_rainfall.csv
#
# RULES:
#   1. Input is already station-to-groundwater-block matched.
#   2. Group records by normalized station name + date.
#   3. If values conflict within a group, retain the MINIMUM rainfall value.
#   4. If values agree, retain one record and log duplicate removal.
#   5. Keep high rainfall values when there is no conflicting value.
#   6. Preserve source station labels and source provenance.
#   7. Keep original input files unchanged.
#
# OUTPUT:
#   cleaned_daily_rainfall.csv
#   rainfall_decision_log.csv
#   conflict_group_summary.csv
#   invalid_input_rows.csv
#   processing_summary.txt
# =============================================================================

from pathlib import Path
from datetime import datetime
import re
import sys
import traceback

import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 1. PATHS
# -----------------------------------------------------------------------------

INPUT_CSV = Path(
    "/content/drive/MyDrive/gw_framework/data/"
    "Rainfall_Rajasthan/Matched_Block_Rainfall/"
    "matched_daily_rainfall.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/gw_framework/data/"
    "Rainfall_Rajasthan/Rainfall_Audit_Minimum_Conflict_Rule"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = OUTPUT_DIR / "cleaned_daily_rainfall.csv"
DECISION_LOG_CSV = OUTPUT_DIR / "rainfall_decision_log.csv"
CONFLICT_SUMMARY_CSV = OUTPUT_DIR / "conflict_group_summary.csv"
INVALID_ROWS_CSV = OUTPUT_DIR / "invalid_input_rows.csv"
SUMMARY_TXT = OUTPUT_DIR / "processing_summary.txt"

print("=" * 78)
print("DAILY RAINFALL CONFLICT RESOLUTION")
print("=" * 78)
print(f"Input : {INPUT_CSV}")
print(f"Output: {OUTPUT_DIR}")
print()

# -----------------------------------------------------------------------------
# 2. CHECK INPUT
# -----------------------------------------------------------------------------

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Input CSV was not found:\n{INPUT_CSV}\n\n"
        "Check that Google Drive is mounted and the file exists."
    )

df = pd.read_csv(INPUT_CSV, low_memory=False)

print(f"Input rows: {len(df):,}")
print("Input columns:")
print(list(df.columns))
print()

# The actual columns reported by your diagnostic cell are used here.
REQUIRED_COLUMNS = [
    "date",
    "rainfall_mm",
    "station_original",
    "station_normalized_for_matching",
    "groundwater_block_match",
]

missing_columns = [c for c in REQUIRED_COLUMNS if c not in df.columns]

if missing_columns:
    raise KeyError(
        f"Required columns missing from input CSV: {missing_columns}\n"
        f"Available columns: {list(df.columns)}"
    )

# Preserve original row order and source record identity for audit purposes.
df = df.copy()
df.insert(0, "input_row_number", np.arange(2, len(df) + 2))

# -----------------------------------------------------------------------------
# 3. NORMALIZE KEYS AND VALIDATE ROWS
# -----------------------------------------------------------------------------

# Keep original column values intact; create separate internal key columns.
df["_station_key"] = (
    df["station_normalized_for_matching"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# If the normalized matching column is empty, fall back to the original label.
fallback = (
    df["_station_key"].isna()
    | df["_station_key"].eq("")
    | df["_station_key"].eq("nan")
)

df.loc[fallback, "_station_key"] = (
    df.loc[fallback, "station_original"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Normalize date to YYYY-MM-DD for reliable grouping.
parsed_dates = pd.to_datetime(df["date"], errors="coerce")
df["_date_key"] = parsed_dates.dt.strftime("%Y-%m-%d")

# Convert rainfall to numeric. Non-numeric values are flagged rather than
# silently treated as zero.
df["_rainfall_numeric"] = pd.to_numeric(
    df["rainfall_mm"], errors="coerce"
)

# Require a station key, valid date and finite, non-negative rainfall value.
# Zero rainfall is valid. High rainfall values are NOT removed by magnitude.
valid_station = (
    df["_station_key"].notna()
    & df["_station_key"].astype(str).str.strip().ne("")
    & df["_station_key"].astype(str).ne("nan")
)

valid_date = df["_date_key"].notna()

valid_rainfall = (
    df["_rainfall_numeric"].notna()
    & np.isfinite(df["_rainfall_numeric"])
    & df["_rainfall_numeric"].ge(0)
)

valid_mask = valid_station & valid_date & valid_rainfall

invalid_df = df.loc[~valid_mask].copy()

def invalid_reason(row):
    reasons = []

    if (
        pd.isna(row["_station_key"])
        or str(row["_station_key"]).strip() in ("", "nan", "<NA>")
    ):
        reasons.append("MISSING_STATION_KEY")

    if pd.isna(row["_date_key"]):
        reasons.append("INVALID_OR_MISSING_DATE")

    value = row["_rainfall_numeric"]

    if pd.isna(value):
        reasons.append("NON_NUMERIC_OR_MISSING_RAINFALL")
    elif not np.isfinite(value):
        reasons.append("NON_FINITE_RAINFALL")
    elif value < 0:
        reasons.append("NEGATIVE_RAINFALL")

    return ";".join(reasons) if reasons else "INVALID_ROW"

if not invalid_df.empty:
    invalid_df["invalid_reason"] = invalid_df.apply(
        invalid_reason, axis=1
    )

# Save invalid rows with original source columns plus diagnostic fields.
invalid_export_columns = [
    c for c in df.columns
    if not c.startswith("_")
]

if not invalid_df.empty:
    invalid_export = invalid_df[
        invalid_export_columns
    ].copy()

    invalid_export["invalid_reason"] = invalid_df["invalid_reason"]
else:
    invalid_export = pd.DataFrame(
        columns=invalid_export_columns + ["invalid_reason"]
    )

invalid_export.to_csv(INVALID_ROWS_CSV, index=False)

work = df.loc[valid_mask].copy()

if work.empty:
    raise ValueError(
        "No valid rainfall records remain after basic validation. "
        f"Review {INVALID_ROWS_CSV} and the input CSV."
    )

work["_rainfall_numeric"] = work["_rainfall_numeric"].astype(float)

print(f"Valid rows:   {len(work):,}")
print(f"Invalid rows: {len(invalid_df):,}")
print()

# -----------------------------------------------------------------------------
# 4. GROUP BY NORMALIZED STATION + DATE
# -----------------------------------------------------------------------------

GROUP_COLUMNS = ["_station_key", "_date_key"]

grouped = work.groupby(
    GROUP_COLUMNS,
    dropna=False,
    sort=False
)

# Summary statistics for each station-date group.
group_stats = grouped.agg(
    input_record_count=("_rainfall_numeric", "size"),
    distinct_rainfall_values=("_rainfall_numeric", "nunique"),
    minimum_rainfall_mm=("_rainfall_numeric", "min"),
    maximum_rainfall_mm=("_rainfall_numeric", "max"),
).reset_index()

group_stats["has_conflict"] = (
    group_stats["distinct_rainfall_values"] > 1
)

group_stats["has_duplicate_records"] = (
    group_stats["input_record_count"] > 1
)

# -----------------------------------------------------------------------------
# 5. BUILD CONFLICT GROUP SUMMARY
# -----------------------------------------------------------------------------

# Keep all source station labels and rainfall values in the group summary
# so each conflict can be traced back to the source records.
conflict_groups = group_stats.loc[
    group_stats["has_conflict"]
].copy()

if not conflict_groups.empty:
    source_details = (
        work.groupby(GROUP_COLUMNS, dropna=False, sort=False)
        .agg(
            source_station_labels=(
                "station_original",
                lambda s: " | ".join(
                    sorted(set(s.dropna().astype(str)))
                )
            ),
            groundwater_block_names=(
                "groundwater_block_match",
                lambda s: " | ".join(
                    sorted(set(s.dropna().astype(str)))
                )
            ),
            source_files=(
                "pdf_file",
                lambda s: " | ".join(
                    sorted(set(s.dropna().astype(str)))
                ) if "pdf_file" in work.columns else ""
            ) if "pdf_file" in work.columns else (
                "station_original",
                lambda s: ""
            ),
            source_pages=(
                "page_number",
                lambda s: " | ".join(
                    sorted(set(s.dropna().astype(str)))
                ) if "page_number" in work.columns else ""
            ) if "page_number" in work.columns else (
                "station_original",
                lambda s: ""
            ),
            rainfall_values_mm=(
                "_rainfall_numeric",
                lambda s: " | ".join(
                    map(
                        str,
                        sorted(set(s.astype(float).tolist()))
                    )
                )
            ),
        )
        .reset_index()
    )

    conflict_groups = conflict_groups.merge(
        source_details,
        on=GROUP_COLUMNS,
        how="left",
        validate="one_to_one"
    )

    conflict_groups = conflict_groups.rename(
        columns={
            "_station_key": "station_normalized_key",
            "_date_key": "date",
        }
    )

else:
    conflict_groups = pd.DataFrame(
        columns=[
            "station_normalized_key",
            "date",
            "input_record_count",
            "distinct_rainfall_values",
            "minimum_rainfall_mm",
            "maximum_rainfall_mm",
            "has_conflict",
            "has_duplicate_records",
            "source_station_labels",
            "groundwater_block_names",
            "source_files",
            "source_pages",
            "rainfall_values_mm",
        ]
    )

conflict_groups.to_csv(CONFLICT_SUMMARY_CSV, index=False)

print(f"Unique station-date groups: {len(group_stats):,}")
print(f"Groups with conflicting values: {int(group_stats['has_conflict'].sum()):,}")
print(f"Groups with duplicate records: {int(group_stats['has_duplicate_records'].sum()):,}")
print()

# -----------------------------------------------------------------------------
# 6. SELECT THE RETAINED RECORD FOR EACH GROUP
# -----------------------------------------------------------------------------

# Sort by:
#   1. normalized station key
#   2. date
#   3. rainfall ascending
#   4. original input order
#
# The first record in each group therefore has the minimum rainfall.
# If several records share that minimum, the first source record is retained.
work = work.sort_values(
    by=[
        "_station_key",
        "_date_key",
        "_rainfall_numeric",
        "input_row_number",
    ],
    ascending=[True, True, True, True],
    kind="mergesort",
)

retained_mask = ~work.duplicated(
    subset=GROUP_COLUMNS,
    keep="first"
)

retained_df = work.loc[retained_mask].copy()

# Build a lookup of the selected value per station-date group.
minimum_lookup = (
    work.groupby(GROUP_COLUMNS, dropna=False, sort=False)
    ["_rainfall_numeric"]
    .min()
    .rename("_minimum_rainfall_mm")
    .reset_index()
)

# This merge uses the actual internal key columns consistently.
# It avoids the previous KeyError caused by referencing "audit_date".
check_df = retained_df.merge(
    minimum_lookup,
    on=GROUP_COLUMNS,
    how="left",
    validate="one_to_one"
)

if not np.allclose(
    check_df["_rainfall_numeric"].to_numpy(dtype=float),
    check_df["_minimum_rainfall_mm"].to_numpy(dtype=float),
    equal_nan=False,
):
    raise AssertionError(
        "Integrity check failed: a retained rainfall value is not "
        "the minimum value for its station-date group."
    )

# -----------------------------------------------------------------------------
# 7. CREATE A ROW-BY-ROW DECISION LOG
# -----------------------------------------------------------------------------

# Map the group-level minimum and group stats to every valid input row.
work_with_stats = work.merge(
    group_stats,
    on=GROUP_COLUMNS,
    how="left",
    validate="many_to_one",
    suffixes=("", "_group"),
)

# The original input row is retained if it is the first record after sorting.
# Use input_row_number to identify the exact selected source row.
retained_input_numbers = set(
    retained_df["input_row_number"].astype(int).tolist()
)

minimum_by_group = {
    (row["_station_key"], row["_date_key"]):
        float(row["minimum_rainfall_mm"])
    for _, row in group_stats.iterrows()
}

def decision_for_row(row):
    key = (row["_station_key"], row["_date_key"])
    minimum_value = minimum_by_group[key]
    value = float(row["_rainfall_numeric"])

    group_count = int(row["input_record_count"])
    distinct_count = int(row["distinct_rainfall_values"])

    if int(row["input_row_number"]) in retained_input_numbers:
        if distinct_count > 1 and value == minimum_value:
            return "RETAINED_MINIMUM_FROM_CONFLICT"
        if distinct_count > 1:
            # This branch should not occur for a correctly selected minimum.
            return "REVIEW_UNEXPECTED_RETAINED_VALUE"
        if group_count > 1:
            return "RETAINED_ONE_OF_IDENTICAL_DUPLICATES"
        return "RETAINED_UNCONFLICTED_VALUE"

    if distinct_count > 1 and value > minimum_value:
        return "SKIPPED_HIGHER_VALUE_IN_CONFLICT"

    if distinct_count == 1:
        return "SKIPPED_IDENTICAL_DUPLICATE"

    return "SKIPPED_DUPLICATE_RECORD"

work_with_stats["decision"] = work_with_stats.apply(
    decision_for_row, axis=1
)

work_with_stats["group_minimum_rainfall_mm"] = (
    work_with_stats["_minimum_rainfall_mm"]
    if "_minimum_rainfall_mm" in work_with_stats.columns
    else work_with_stats["_rainfall_numeric"]
)

# Ensure group minimum is available directly from the lookup.
if "minimum_rainfall_mm" in work_with_stats.columns:
    work_with_stats["group_minimum_rainfall_mm"] = (
        work_with_stats["minimum_rainfall_mm"]
    )
else:
    work_with_stats["group_minimum_rainfall_mm"] = work_with_stats.apply(
        lambda row: minimum_by_group[
            (row["_station_key"], row["_date_key"])
        ],
        axis=1
    )

# Restore log to original input order for easier auditing.
work_with_stats = work_with_stats.sort_values(
    "input_row_number", kind="mergesort"
)

# Include all source columns and useful audit columns.
log_columns = [
    c for c in df.columns
    if not c.startswith("_")
]

decision_log = work_with_stats[log_columns].copy()
decision_log["station_normalized_key"] = (
    work_with_stats["_station_key"].to_numpy()
)
decision_log["audit_date"] = (
    work_with_stats["_date_key"].to_numpy()
)
decision_log["rainfall_numeric_mm"] = (
    work_with_stats["_rainfall_numeric"].to_numpy()
)
decision_log["group_minimum_rainfall_mm"] = (
    work_with_stats["group_minimum_rainfall_mm"].to_numpy()
)
decision_log["input_record_count_in_group"] = (
    work_with_stats["input_record_count"].to_numpy()
)
decision_log["distinct_values_in_group"] = (
    work_with_stats["distinct_rainfall_values"].to_numpy()
)
decision_log["decision"] = work_with_stats["decision"].to_numpy()

# Add invalid rows to the audit log as well, so every input row is accounted for.
if not invalid_df.empty:
    invalid_log = invalid_df[log_columns].copy()
    invalid_log["station_normalized_key"] = (
        invalid_df["_station_key"].astype("string").to_numpy()
    )
    invalid_log["audit_date"] = invalid_df["_date_key"].to_numpy()
    invalid_log["rainfall_numeric_mm"] = (
        invalid_df["_rainfall_numeric"].to_numpy()
    )
    invalid_log["group_minimum_rainfall_mm"] = np.nan
    invalid_log["input_record_count_in_group"] = np.nan
    invalid_log["distinct_values_in_group"] = np.nan
    invalid_log["decision"] = (
        invalid_df["invalid_reason"].to_numpy()
    )

    decision_log = pd.concat(
        [decision_log, invalid_log],
        ignore_index=True
    )

decision_log = decision_log.sort_values(
    "input_row_number", kind="mergesort"
)

decision_log.to_csv(DECISION_LOG_CSV, index=False)

# -----------------------------------------------------------------------------
# 8. CREATE FINAL CLEANED DAILY RAINFALL FILE
# -----------------------------------------------------------------------------

cleaned_df = retained_df.copy()

# Keep the original date column as YYYY-MM-DD.
cleaned_df["date"] = cleaned_df["_date_key"]

# Use the numeric rainfall values selected by the rule.
cleaned_df["rainfall_mm"] = cleaned_df["_rainfall_numeric"]

# Remove internal helper columns but keep provenance and input row number.
cleaned_df = cleaned_df[
    [
        c for c in cleaned_df.columns
        if not c.startswith("_")
    ]
].copy()

# Ensure stable order in the final output.
cleaned_df = cleaned_df.sort_values(
    by=[
        "station_normalized_for_matching",
        "date",
        "input_row_number",
    ],
    kind="mergesort"
)

cleaned_df.to_csv(CLEANED_CSV, index=False)

# -----------------------------------------------------------------------------
# 9. FINAL INTEGRITY CHECKS
# -----------------------------------------------------------------------------

# Each normalized station-date should occur exactly once.
duplicate_final = cleaned_df.duplicated(
    subset=[
        "station_normalized_for_matching",
        "date",
    ],
    keep=False
)

# The authoritative uniqueness key used internally is _station_key + _date_key.
# Check that too, since original station labels can vary.
final_keys = retained_df[GROUP_COLUMNS].copy()
duplicate_internal = final_keys.duplicated(
    subset=GROUP_COLUMNS,
    keep=False
)

if duplicate_internal.any():
    raise AssertionError(
        "Final data still contains duplicate normalized station-date keys."
    )

if duplicate_final.any():
    print(
        "Note: original station labels differ within some normalized "
        "station-date groups. Internal normalized-key uniqueness passed."
    )

if len(cleaned_df) != len(group_stats):
    raise AssertionError(
        f"Output row count ({len(cleaned_df)}) does not equal the number "
        f"of valid normalized station-date groups ({len(group_stats)})."
    )

# Confirm retained values are the group minima.
final_value_check = retained_df.merge(
    minimum_lookup,
    on=GROUP_COLUMNS,
    how="left",
    validate="one_to_one"
)

if not np.allclose(
    final_value_check["_rainfall_numeric"].to_numpy(dtype=float),
    final_value_check["_minimum_rainfall_mm"].to_numpy(dtype=float),
):
    raise AssertionError(
        "Final integrity check failed: retained values differ from group minima."
    )

# -----------------------------------------------------------------------------
# 10. PROCESSING SUMMARY
# -----------------------------------------------------------------------------

conflict_group_count = int(group_stats["has_conflict"].sum())
duplicate_group_count = int(
    (group_stats["input_record_count"] > 1).sum()
)
same_value_duplicate_groups = int(
    (
        (group_stats["input_record_count"] > 1)
        & (group_stats["distinct_rainfall_values"] == 1)
    ).sum()
)

skipped_higher_values = int(
    (decision_log["decision"] == "SKIPPED_HIGHER_VALUE_IN_CONFLICT").sum()
)

skipped_identical_duplicates = int(
    (decision_log["decision"] == "SKIPPED_IDENTICAL_DUPLICATE").sum()
)

retained_conflict_minima = int(
    (decision_log["decision"] == "RETAINED_MINIMUM_FROM_CONFLICT").sum()
)

summary_lines = [
    "DAILY RAINFALL CONFLICT RESOLUTION SUMMARY",
    "=" * 55,
    f"Run timestamp: {datetime.now().isoformat(timespec='seconds')}",
    f"Input file: {INPUT_CSV}",
    f"Output directory: {OUTPUT_DIR}",
    "",
    f"Input records: {len(df):,}",
    f"Valid input records: {len(work):,}",
    f"Invalid input records: {len(invalid_df):,}",
    f"Unique normalized station-date groups: {len(group_stats):,}",
    f"Final cleaned records: {len(cleaned_df):,}",
    "",
    f"Groups with conflicting rainfall values: {conflict_group_count:,}",
    f"Groups containing duplicate records: {duplicate_group_count:,}",
    f"Groups with identical duplicate values: {same_value_duplicate_groups:,}",
    f"Higher conflicting records skipped: {skipped_higher_values:,}",
    f"Identical duplicate records skipped: {skipped_identical_duplicates:,}",
    f"Retained minimum values from conflict groups: {retained_conflict_minima:,}",
    "",
    "Cleaning rules applied:",
    "1. Group by normalized station key and date.",
    "2. Keep the minimum rainfall value within a conflicting group.",
    "3. Collapse identical duplicates to one record.",
    "4. Retain non-conflicting values regardless of magnitude.",
    "5. Do not alter source PDFs, groundwater Excel files, or input CSV.",
    "",
    "Integrity checks:",
    "1. Each internal normalized station-date key occurs once.",
    "2. Each retained value equals the minimum value for its group.",
    "3. Final row count equals valid normalized station-date group count.",
    "",
    "Output files:",
    f"  {CLEANED_CSV.name}",
    f"  {DECISION_LOG_CSV.name}",
    f"  {CONFLICT_SUMMARY_CSV.name}",
    f"  {INVALID_ROWS_CSV.name}",
    f"  {SUMMARY_TXT.name}",
]

summary_text = "\n".join(summary_lines)

with open(SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(summary_text)

# -----------------------------------------------------------------------------
# 11. DISPLAY RESULTS
# -----------------------------------------------------------------------------

print("=" * 78)
print("PROCESSING COMPLETED")
print("=" * 78)
print(summary_text)
print()
print("Saved output files:")
for path in [
    CLEANED_CSV,
    DECISION_LOG_CSV,
    CONFLICT_SUMMARY_CSV,
    INVALID_ROWS_CSV,
    SUMMARY_TXT,
]:
    print(f"  {path}  [{'OK' if path.exists() else 'MISSING'}]")

print()
print("Preview of cleaned data:")
display(pd.read_csv(CLEANED_CSV).head(10))

print("Preview of conflict groups:")
if CONFLICT_SUMMARY_CSV.exists():
    conflict_preview = pd.read_csv(CONFLICT_SUMMARY_CSV)
    display(conflict_preview.head(10))
else:
    print("No conflict summary file was generated.")

print()
print("All done.")

**[orig #36]** Check columns of `matched_daily_rainfall.csv`  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd

input_csv = (
    "/content/drive/MyDrive/gw_framework/data/"
    "Rainfall_Rajasthan/Matched_Block_Rainfall/"
    "matched_daily_rainfall.csv"
)

df = pd.read_csv(input_csv)

print("Input CSV columns:")
for col in df.columns:
    print(f"  {col!r}")

print("\nPossible date columns:")
for col in df.columns:
    if "date" in col.lower() or "day" in col.lower():
        print(f"  {col!r}")

print("\nFirst 5 rows:")
display(df.head())